#### <B>Agents & Tools

Agents: Agents enable llms to work with tools that perform various task.<Br>
We required these few things for using agents:<br>
1. A base llm<br>
2. A tool that will be interacting with,<br>
3. An agent to control th interaction

In [11]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()
from langchain_classic import LLMMathChain
from langchain_classic.agents import Tool
import os

In [16]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", google_api_key = os.getenv("GOOGLE_API_KEY"), temperature =0.5)

In [17]:
# initialize the math tool
llm_math = LLMMathChain(llm=llm)
math_tool = Tool(
    name="Calculator",
    func=llm_math.run,
    description="Useful for when you need to answer questions about math"

)

# when giving tools to llma we must pass the list
tools = [math_tool] # list of all tools agent will use



C:\Users\HP\AppData\Local\Temp\ipykernel_29188\1279420715.py:2: UserWarning: Directly instantiating an LLMMathChain with an llm is deprecated. Please instantiate with llm_chain argument or using the from_llm class method.
  llm_math = LLMMathChain(llm=llm)


#### Now creating agent

#### Zero Shot ReAct

It is an agent that can create realistic context even without being trained on a specific data. It can be used for various task such as generating creative text formats, language translation, and generating different types of creative content.

In [18]:
from langchain_classic.agents import initialize_agent

zero_shot_agent = initialize_agent(
    agent = "zero-shot-react-description",
    tools=tools,
    llm=llm,
    verbose = True,
    max_iterations = 10
)


In [19]:
zero_shot_agent("What os root over 25")



> Entering new AgentExecutor chain...
Action: Calculator
Action Input: sqrt(25)
Observation: Answer: 5.0
Thought:I now know the final answer
Final Answer: 5.0

> Finished chain.


{'input': 'What os root over 25', 'output': '5.0'}

In [20]:
problem = """
        You are building a house. There are two bedrooms of 5 metres by 5 metres each and drawing cum open kitchen 
        is 7 metres by 6 metres and balcony of 3 metres by 2 metres. 
        What is the total area of your house?
        """
zero_shot_agent(problem)



> Entering new AgentExecutor chain...
Action: Calculator
Action Input: (5 * 5) * 2 + (7 * 6) + (3 * 2)
Observation: Answer: 98
Thought:I now know the final answer
Final Answer: The total area of your house is 98 square metres.

> Finished chain.


{'input': '\n        You are building a house. There are two bedrooms of 5 metres by 5 metres each and drawing cum open kitchen \n        is 7 metres by 6 metres and balcony of 3 metres by 2 metres. \n        What is the total area of your house?\n        ',
 'output': 'The total area of your house is 98 square metres.'}

#### Using Multiple Tools

In [22]:
from langchain_classic.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search_tool = Tool.from_function(
    func=search.run,
    name="Search",
    description="Useful when we need to search the internet for information"
)

llm_math_chain = LLMMathChain(llm = llm, verbose=True)

math_tool = Tool.from_function(
    func = llm_math_chain.run,
    name="Calculator",
    description="Useful for when we asked to perform math calculations"
)


C:\Users\HP\AppData\Local\Temp\ipykernel_29188\1281161026.py:11: UserWarning: Directly instantiating an LLMMathChain with an llm is deprecated. Please instantiate with llm_chain argument or using the from_llm class method.
  llm_math_chain = LLMMathChain(llm = llm, verbose=True)


In [23]:
# define the agent

tools = [search_tool, math_tool]

agent = initialize_agent(
    tools,
    llm,
    agent="zero-shot-react-description",
    verbose = True
)



In [24]:
agent.run("""Get Microsoft Stock Price taken from Google Finance and display in both USD and INR values""")

C:\Users\HP\AppData\Local\Temp\ipykernel_29188\3788483621.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  agent.run("""Get Microsoft Stock Price taken from Google Finance and display in both USD and INR values""")




> Entering new AgentExecutor chain...
Action: Search
Action Input: Microsoft stock price Google Finance
Observation: Based on 37 Wall Street analysts offering 12 month price targets for MSFT in the last 3 months. Major U.S. stock indexes fell sharply on Wednesday after the Federal Reserve signaled that interest rate hikes may be necessary later this year. 17 hours ago · View real-time stock prices and stock quotes for a full financial overview. 1 day ago · Should You Buy or Sell Microsoft Stock? Get The Latest MSFT Stock Analysis, Price Target, Dividend Info, Headlines, and Short Interest at MarketBeat. Jun 8, 2026 · The Investor Relations website contains information about Microsoft Corporation's business for stockholders, potential investors, and financial analysts. View the MSFT premarket stock price ahead of the market session or assess the after hours quote. Monitor the latest movements within the Microsoft Corporation real time stock price... What is the price of Microsoft stoc

'The Microsoft stock price is 426.99 USD. In Indian Rupees (INR), this is approximately 38536.27 INR.'

#### Custom Tools

In [25]:
from langchain_classic.tools import BaseTool,tool
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate


@tool("JSON_Response", return_direct=True)
def StructuresResponseTool(question: str):
    """
    Use this tool to send a prompt and get a json returned
    with three fields - Topic, Question_Details and Detailed_Response
    """
    json_prompt = PromptTemplate.from_template(
        """
        Return a json response with an 'answer' key that answers the following question: {question}.
        The JSON object will have three fields - Topic, Question_Details and Detailed_Response
    """)
    llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash", google_api_key = os.getenv("GOOGLE_API_KEY"), temperature =0.5)

    json_parser = JsonOutputParser()
    json_chain = json_prompt | llm | json_parser
    x = json_chain.invoke({"question" : question})
    return x


In [26]:
tools = [StructuresResponseTool]
zero_shot_agent = initialize_agent(
    agent = "zero-shot-react-description",
    tools=tools,
    llm = llm,
    verbose = True,
    max_iteration = 10
)